# Extract many layers as a dict

Use `tl.extract` when you want a compact dictionary of activations for a named set of layers.

In [ ]:
from pathlib import Path
import importlib.util
import sys

import torch
import torchlens as tl

shared_path = None
for root in [Path.cwd(), *Path.cwd().parents]:
    candidate = root / "notebooks" / "total_audit" / "_shared.py"
    if candidate.exists():
        shared_path = candidate
        break
if shared_path is None:
    raise FileNotFoundError("Could not find notebooks/total_audit/_shared.py")

spec = importlib.util.spec_from_file_location("torchlens_total_audit_shared", shared_path)
if spec is None or spec.loader is None:
    raise ImportError(f"Could not load {shared_path}")
shared = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = shared
spec.loader.exec_module(shared)
tiny_model = shared.tiny_model

torch.manual_seed(0)
model = tiny_model(seed=7).eval()
x = torch.randn(2, 4)

Pass either a list of layer labels or a mapping from your names to TorchLens labels.

In [ ]:
activations = tl.extract(
    model, x, {"hidden": "linear_1_1", "relu": "relu_1_2", "logits": "linear_2_3"}
)
print(activations.keys())
{k: tuple(v.shape) for k, v in activations.items()}

The values are ordinary tensors, so this dict can feed metrics, probes, caches, or plotting code.

In [ ]:
summary = {name: round(tensor.norm().item(), 4) for name, tensor in activations.items()}
summary